# 01 · Bronze — Auto Loader ingest

Incrementally ingests raw landing CSVs into the bronze Delta table using **Auto
Loader** (`cloudFiles`), producing a stable, append-only history of trips.

- Maps every historical Citibike column-naming variant onto one **canonical schema**
  (see [`shared/canonical.py`](../../shared/canonical.py)).
- Synthesises a **deterministic `ride_id`** (SHA-256 of the natural key) for legacy
  files that predate it, so re-processing is idempotent.
- Adds run **metadata** and enables **Change Data Feed** for the silver layer.

**Upstream:** `00_landing` · **Downstream:** `02_silver`

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

from pyspark.sql.functions import coalesce, col, concat_ws, create_map, lit, sha2

# Reuse the shared, unit-tested canonicalization rules (see src/shared/canonical.py)
sys.path.append(str(Path.cwd().parent.parent))
from shared.canonical import missing_columns, resolve_column_groups

# Change Data Feed is required so the silver layer can consume row-level changes.
spark.conf.set("spark.databricks.delta.properties.defaults.enableChangeDataFeed", "true")

env_name = dbutils.widgets.get("env") if dbutils.widgets.get("env") else "dev"
pipeline_id = dbutils.widgets.get("pipeline_id") if dbutils.widgets.get("pipeline_id") else "placeholder"
job_run_id = dbutils.widgets.get("job_run_id") if dbutils.widgets.get("job_run_id") else "placeholder"
task_id = dbutils.widgets.get("task_run_id") if dbutils.widgets.get("task_run_id") else "placeholder"

catalog_name = f"citibike_ext_{env_name}" if env_name == "dev" else f"citibike_{env_name}"
folder_path = f"/Volumes/{catalog_name}/00_landing/source_data/citibike_trips/"
checkpoint_path = f"/Volumes/{catalog_name}/01_bronze/_checkpoint/citibike_trips/"
schema_path = f"/Volumes/{catalog_name}/01_bronze/_schema/citibike_trips/"
table_path = f"{catalog_name}.01_bronze.citibike_trips"


def to_canonical(df):
    """Rename/merge every known source column spelling onto the canonical schema."""
    groups = resolve_column_groups(df.columns)
    return df.select(
        *[
            (coalesce(*[col(c) for c in srcs]) if len(srcs) > 1 else col(srcs[0])).alias(canon)
            for canon, srcs in groups.items()
        ]
    )


def add_missing_columns(df):
    """Add expected-but-absent columns as typed NULLs so the bronze schema stays stable."""
    return df.select(
        "*",
        *[lit(None).cast(t).alias(c) for c, t in missing_columns(df.columns).items()],
    )


df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .option("rescuedDataColumn", "_rescued_data")
    .load(folder_path)
)

df = to_canonical(df=df)

# If ride_id is absent from this batch's schema, introduce it as NULL so the
# coalesce below can synthesise a deterministic id for legacy files.
if "ride_id" not in df.columns:
    df = df.withColumn("ride_id", lit(None).cast("string"))

df = add_missing_columns(df)

# Older files predate ride_id — synthesise a stable, deterministic id by hashing
# the natural key so re-processing the same trip yields the same id (idempotent).
df = df.withColumn(
    "ride_id",
    coalesce(
        col("ride_id"),
        sha2(
            concat_ws(
                "||",
                col("started_at"),
                coalesce(col("start_station_id"), lit("∅")),
                coalesce(col("end_station_id"), lit("∅")),
                coalesce(col("bike_id"), lit("∅")),
                coalesce(col("trip_duration"), lit("∅")),
            ),
            256,
        ),
    ),
)

df = df.withColumn(
    "metadata",
    create_map(
        lit("pipeline_id"),
        lit(pipeline_id),
        lit("run_id"),
        lit(job_run_id),
        lit("task_id"),
        lit(task_id),
        lit("processed_timestamp"),
        lit(datetime.now().isoformat()),
    ),
)

df = df.select(
    "ride_id",
    "bike_id",
    "rideable_type",
    "started_at",
    "start_station_id",
    "start_station_name",
    "ended_at",
    "end_station_id",
    "end_station_name",
    "start_lat",
    "start_lng",
    "end_lat",
    "end_lng",
    "member_casual",
    "birth_year",
    "gender",
    "trip_duration",
    "metadata",
    "_rescued_data",
)

(
    df.writeStream.option("checkpointLocation", checkpoint_path)
    .option("delta.enableChangeDataFeed", "true")
    .trigger(availableNow=True)
    .outputMode("append")
    .toTable(table_path)
)

In [ ]:
# One-time setup (run outside this notebook): grant the Unity Catalog Access
# Connector's managed identity the roles it needs to read the external storage
# account and to receive Auto Loader file-notification events. IDs below are
# placeholders — substitute your own subscription / resource group / storage.
#
# PRINCIPAL_ID="<access-connector-managed-identity-object-id>"
# STORAGE_ID="/subscriptions/<sub-id>/resourceGroups/<rg>/providers/Microsoft.Storage/storageAccounts/<account>"
#
# for ROLE in \
#   "Storage Account Contributor" \
#   "Storage Queue Data Contributor" \
#   "EventGrid EventSubscription Contributor" \
#   "Storage Blob Data Contributor"; do
#   az role assignment create \
#     --assignee-object-id "$PRINCIPAL_ID" \
#     --assignee-principal-type ServicePrincipal \
#     --role "$ROLE" \
#     --scope "$STORAGE_ID"
# done